# AML-Graph MVP — explainable detection of laundering schemes

Hybrid **interpretable motif-mining + GNN edge scoring + LLM narrative** on IBM AMLworld (HI-Small).
Priorities: correct minority-class metrics, strict temporal validation, and explainability.

Heavy logic lives in `src/`; this notebook narrates the pipeline stage by stage.

## 0. Setup — dependencies, seeds, paths

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src import config, data_io, graph_build, metrics, viz, pipeline
from src.config import DEFAULT_RUN

config.set_seeds()
config.ensure_dirs()
cfg = DEFAULT_RUN
print('seed', config.SEED, '| sample_edges', cfg.sample_edges)

## 1. Data — IBM AMLworld HI-Small
Downloads via Kaggle CLI (`ealtman2019/ibm-transactions-for-anti-money-laundering-aml`) if absent, then parses the ground-truth `HI-Small_Patterns.txt` into typed motif labels.

In [ ]:
df, patterns = pipeline.load_and_prepare(cfg)
print('transactions:', df.height, '| illicit rate:', df.get_column("is_laundering").mean())
data_io.patterns_summary(patterns)

## 2. EDA — imbalance, degree, volume, formats  →  `fig01_eda_imbalance.png`

In [ ]:
viz.fig01_eda(df)
print('Extreme imbalance (<1% illicit) -> we report minority-F1 & PR-AUC, never accuracy/ROC-AUC.')
from IPython.display import Image
Image(str(config.FIGURES / 'fig01_eda_imbalance.png'))

## 3. Directed temporal multigraph (node = Bank_Account, edge = transaction)

In [ ]:
g = graph_build.build_graph(df)
print('nodes:', g.number_of_nodes(), '| edges:', g.number_of_edges())
ds = pipeline.make_dataset(g, cfg)
print('edge features:', ds.feature_names)

## 4. Motif mining — fan-out / fan-in / cycle / gather-scatter / scatter-gather
Explicit, unit-tested detectors (`src/motifs.py`, `tests/test_motifs.py`). →  `fig02_motif_freq.png`

In [ ]:
participation = pipeline.motif_participation(ds)
for m, (ill, lic) in participation.items():
    print(f'{m:16s} illicit={ill:6d}  licit={lic:6d}')
viz.fig02_motif_freq(participation)
Image(str(config.FIGURES / 'fig02_motif_freq.png'))

## 5. Baseline (LightGBM) + GNN (GraphSAGE) — strict temporal split
Train on early timestamps, test on strictly later ones. Message passing uses **train-only** edges, so no future information leaks into node embeddings (see `tests/test_no_leakage.py`).

In [ ]:
scores, y_test, test_edge_ids = pipeline.run_models(ds, cfg)
gt_motif_map = pipeline.groundtruth_motif_map(patterns, ds)
print('test edges:', y_test.size, '| illicit in test:', int(y_test.sum()))

## 6. Results — PR-AUC, minority-F1, precision@k, alert-reduction  →  `fig03`, `fig04`, `metrics.json`

In [ ]:
result = pipeline.compute_metrics(scores, y_test, test_edge_ids, gt_motif_map, cfg)
viz.fig03_pr_curve(y_test, scores)
pk = {n: metrics.precision_at_k(y_test, s, cfg.precision_at_k) for n, s in scores.items()}
viz.fig04_precision_at_k(pk)
for m in ('LightGBM', 'GNN'):
    r = result[m]
    print(f"{m:9s} F1={r['minority_f1']:.3f} PR-AUC={r['pr_auc']:.3f} "
          f"alert_reduction@rec{cfg.fixed_recall}={r['alert_reduction']:.2f}")
Image(str(config.FIGURES / 'fig03_pr_curve.png'))

In [ ]:
Image(str(config.FIGURES / 'fig04_precision_at_k.png'))

## 7. Trace + auto-narrative — a concrete ground-truth chain  →  `fig05`, `fig06`
Pick a labelled pattern, light up the illicit chain against grey traffic, and generate a SAR-style narrative (template by default; set `use_llm=True` for the Anthropic call).

In [ ]:
from src.narrative import template_narrative
chain_edges, facts = pipeline.pick_trace(g, patterns, ds, scores['GNN'], test_edge_ids)
viz.fig05_ring_trace(g, chain_edges, f'Ground-truth {facts.motif_type} chain')
narrative = template_narrative(facts)   # llm_narrative(facts) for the Anthropic version
print(narrative)
viz.fig06_narrative_card(facts, narrative)
Image(str(config.FIGURES / 'fig05_ring_trace.png'))

In [ ]:
pipeline.save_scored_edges(ds, scores['GNN'], test_edge_ids)  # artifact for the Streamlit demo
result['trace_facts'] = facts.__dict__
result['narrative'] = narrative
config.METRICS_JSON.write_text(json.dumps(result, indent=2, ensure_ascii=False))
Image(str(config.FIGURES / 'fig06_narrative_card.png'))

## 8. Streamlit demo
```bash
streamlit run app/streamlit_app.py
```
Enter an account → interactive ego graph with the highlighted chain + narrative.

## Key numbers — baseline vs GNN

In [ ]:
import polars as pl
rows = []
for m in ('LightGBM', 'GNN', 'Random'):
    r = result[m]
    rows.append({
        'model': m,
        'minority_F1': round(r['minority_f1'], 3),
        'PR_AUC': round(r['pr_auc'], 3),
        'p@100': round(r.get('p@100', 0.0), 3),
        'alert_reduction': round(r['alert_reduction'], 3),
    })
pl.DataFrame(rows)